# Version 2 Summary

In Version 2, the system improves the baseline RAG retrieval pipeline by replacing raw L2 distance with cosine similarity over normalized sentence embeddings. It also introduces a cross-encoder reranker to refine the top retrieved chunks before answer generation.

The pipeline now includes:

1. PDF text extraction
2. Page-level metadata construction
3. Text chunking
4. Sentence embedding generation
5. FAISS cosine-similarity retrieval
6. Cross-encoder reranking
7. Evidence context construction
8. Retrieval evaluation using Recall@K and MRR

This version improves retrieval reliability and provides a stronger foundation for evidence-grounded answer generation.

In [1]:
import fitz
import pandas as pd
import numpy as np
from tqdm import tqdm

pdf_path = "Artificial_ Intelligence_Index_Report_2025.pdf"

def extract_pdf_text(pdf_path):
    doc = fitz.open(pdf_path)
    pages = []

    for page_idx, page in enumerate(doc):
        text = page.get_text("text")
        pages.append({
            "page": page_idx + 1,
            "text": text
        })

    return pages

pages = extract_pdf_text(pdf_path)

print("Number of pages:", len(pages))
print(pages[0]["text"][:1000])

Number of pages: 457
Artificial Intelligence
Index Report 2025



In [3]:
def chunk_text(text, chunk_size=800, overlap=150):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()

        if len(chunk) > 100:
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [5]:
all_chunks = []

for page in pages:
    chunks = chunk_text(page["text"], chunk_size=800, overlap=150)

    for chunk_id, chunk in enumerate(chunks):
        all_chunks.append({
            "source": "Stanford AI Index Report 2025",
            "page": page["page"],
            "chunk_id": chunk_id,
            "text": chunk
        })

chunks_df = pd.DataFrame(all_chunks)

print("Total chunks:", len(chunks_df))
chunks_df.head()

Total chunks: 1421


,source,page,chunk_id,text
0,Stanford AI Index Report 2025,2,0,Artificial Intelligence\nIndex Report 2025\n1\...
1,Stanford AI Index Report 2025,2,1,n offshoot of the One Hundred Year Study of Ar...
2,Stanford AI Index Report 2025,2,2,the rapid evolution of underlying technologies...
3,Stanford AI Index Report 2025,2,3,the world. We have briefed companies like Acce...
4,Stanford AI Index Report 2025,3,0,Artificial Intelligence\nIndex Report 2025\n2\...


In [7]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

texts = chunks_df["text"].tolist()

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=False,
    convert_to_numpy=True
)

embeddings.shape

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(1421, 384)

In [9]:
import faiss

# Make sure embeddings are float32
embeddings_cosine = embeddings.astype("float32").copy()

# Normalize embeddings for cosine similarity
faiss.normalize_L2(embeddings_cosine)

dimension = embeddings_cosine.shape[1]

# Inner product on normalized vectors = cosine similarity
index_cosine = faiss.IndexFlatIP(dimension)
index_cosine.add(embeddings_cosine)

print("Number of vectors in cosine index:", index_cosine.ntotal)

Number of vectors in cosine index: 1421


## Write a Cosine Retrieval Function

In [12]:
def retrieve_dense_cosine(query, top_k=10):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        show_progress_bar=False
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = index_cosine.search(query_embedding, top_k)

    results = []

    for rank, idx in enumerate(indices[0]):
        row = chunks_df.iloc[idx]

        results.append({
            "rank": rank + 1,
            "page": int(row["page"]),
            "chunk_id": int(row["chunk_id"]),
            "cosine_score": float(scores[0][rank]),
            "text": row["text"]
        })

    return results

In [24]:
query = "How has AI investment changed in recent years?"

results = retrieve_dense_cosine(query, top_k=5)

for r in results:
    print("=" * 100)
    print(f"Rank: {r['rank']} | Page: {r['page']} | Cosine Score: {r['cosine_score']:.4f}")
    print("-" * 100)
    print(r["text"][:800])

Rank: 1 | Page: 359 | Cosine Score: 0.7905
----------------------------------------------------------------------------------------------------
358
Artificial Intelligence
Index Report 2025
Table of Contents
Chapter 6 Preview
6.3 Public Investment in AI
Chapter 6: Policy and Governance
Figure 6.3.6 illustrates the trends in public AI investment 
over time across two significant regions of AI investment, the 
United States and Europe. Both regions have seen substantial 
growth in AI-related spending over the past decade. Notably, 
Europe’s total AI investment in 2023 was approximately 67 
times higher than in 2013, compared to a fifteenfold increase 
in the United States. Europe experienced particularly sharp 
increases in investment, with a 400% year-over-year increase 
in 2017, followed by another major spike of 200% year-over-
year in 2019—a year that also saw a peak in the number of 
national AI strategies released globally. Th
Rank: 2 | Page: 248 | Cosine Score: 0.7830
------------

## Add Cross-Encoder Reranking

In [18]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

D:\Columbia_University\Columbia_Language\Python\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Agamotto\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

## Dense Retrieval + Reranking Function

In [26]:
def retrieve_with_rerank(query, first_k=20, top_k=5):
    # Step 1: dense retrieval
    candidates = retrieve_dense_cosine(query, top_k=first_k)

    # Step 2: build query-chunk pairs
    pairs = [(query, item["text"]) for item in candidates]

    # Step 3: rerank
    rerank_scores = reranker.predict(pairs)

    for item, score in zip(candidates, rerank_scores):
        item["rerank_score"] = float(score)

    # Step 4: sort by rerank score
    reranked = sorted(
        candidates,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    # Step 5: update rank
    for i, item in enumerate(reranked[:top_k]):
        item["rank"] = i + 1

    return reranked[:top_k]

In [28]:
query = "How has AI investment changed in recent years?"

reranked_results = retrieve_with_rerank(
    query,
    first_k=20,
    top_k=5
)

for r in reranked_results:
    print("=" * 120)
    print(
        f"Rank: {r['rank']} | Page: {r['page']} | "
        f"Cosine: {r['cosine_score']:.4f} | Rerank: {r['rerank_score']:.4f}"
    )
    print("-" * 120)
    print(r["text"][:1000])

Rank: 1 | Page: 248 | Cosine: 0.7830 | Rerank: 5.2594
------------------------------------------------------------------------------------------------------------------------
s investment section 
in the AI Index includes data on generative AI 
investments.
4.3 Investment
Corporate Investment
Figure 4.3.1 illustrates the trend in global corporate AI investment from 
2013 to 2024, including mergers and acquisitions, minority stakes, private 
investments, and public offerings. 
In 2024, the total investment grew to $252.3 billion, an increase of 25.5% 
from 2023. The most significant upturn occurred in private investment, 
which rose by 44.5% compared with the previous year, while mergers and 
acquisitions increased by 12.1%. Over the past decade, AI-related investments 
have increased nearly thirteenfold.
20.06
37.32
25.72
43.1
58.18
73.79
145.4
113.01
104.34
150.79
88.19
24.68
21.89
36.43
39.83
175.36
121.39
82.26
92.19
14.57
19.04
25.43
33.82
53.72
79.62
103.2
Rank: 2 | Page: 249 | Co

## Make a Clean Display Function

In [31]:
def show_reranked_results(query, first_k=20, top_k=5):
    results = retrieve_with_rerank(
        query=query,
        first_k=first_k,
        top_k=top_k
    )

    display_df = pd.DataFrame([
        {
            "rank": r["rank"],
            "page": r["page"],
            "chunk_id": r["chunk_id"],
            "cosine_score": round(r["cosine_score"], 4),
            "rerank_score": round(r["rerank_score"], 4),
            "preview": r["text"][:300].replace("\n", " ")
        }
        for r in results
    ])

    return display_df

In [34]:
show_reranked_results(
    "What are the major trends in AI model performance?",
    first_k=20,
    top_k=5
)

,rank,page,chunk_id,cosine_score,rerank_score,preview
0,1,86,0,0.6169,6.6256,85 Artificial Intelligence Index Report 2025 T...
1,2,15,0,0.6084,5.0061,Artificial Intelligence Index Report 2025 14 3...
2,3,171,1,0.6217,4.4359,"features, as businesses and governments are i..."
3,4,59,0,0.5896,4.0666,Table of Contents 58 Artificial Intelligence I...
4,5,101,1,0.6170,3.6203,"ama models, Anthropic with Claude, High- Flye..."


## Build Evidence Context for Answer Generation

In [37]:
def build_context(results, max_chars=3500):
    context_parts = []
    total_chars = 0

    for i, r in enumerate(results):
        source_header = f"[Source {i+1}] Page {r['page']}, Chunk {r['chunk_id']}\n"
        source_text = r["text"].strip()
        source_block = source_header + source_text

        if total_chars + len(source_block) > max_chars:
            break

        context_parts.append(source_block)
        total_chars += len(source_block)

    return "\n\n".join(context_parts)

In [39]:
query = "How has AI investment changed in recent years?"

results = retrieve_with_rerank(query, first_k=20, top_k=5)
context = build_context(results)

print(context[:3000])

[Source 1] Page 248, Chunk 1
s investment section 
in the AI Index includes data on generative AI 
investments.
4.3 Investment
Corporate Investment
Figure 4.3.1 illustrates the trend in global corporate AI investment from 
2013 to 2024, including mergers and acquisitions, minority stakes, private 
investments, and public offerings. 
In 2024, the total investment grew to $252.3 billion, an increase of 25.5% 
from 2023. The most significant upturn occurred in private investment, 
which rose by 44.5% compared with the previous year, while mergers and 
acquisitions increased by 12.1%. Over the past decade, AI-related investments 
have increased nearly thirteenfold.
20.06
37.32
25.72
43.1
58.18
73.79
145.4
113.01
104.34
150.79
88.19
24.68
21.89
36.43
39.83
175.36
121.39
82.26
92.19
14.57
19.04
25.43
33.82
53.72
79.62
103.2

[Source 2] Page 249, Chunk 0
248
Artificial Intelligence
Index Report 2025
Table of Contents
Chapter 4 Preview
Startup Activity
This section analyzes private investment 

## Build a RAG Prompt

In [42]:
def build_rag_prompt(query, results):
    context = build_context(results)

    prompt = f"""
You are an AI assistant answering questions based on a long AI report.

Use only the provided sources to answer the question.
If the sources do not contain enough information, say that the report context is insufficient.
Cite the source number and page number when possible.

Question:
{query}

Sources:
{context}

Answer:
"""
    return prompt

In [44]:
query = "How has AI investment changed in recent years?"

results = retrieve_with_rerank(query, first_k=20, top_k=5)
prompt = build_rag_prompt(query, results)

print(prompt[:4000])


You are an AI assistant answering questions based on a long AI report.

Use only the provided sources to answer the question.
If the sources do not contain enough information, say that the report context is insufficient.
Cite the source number and page number when possible.

Question:
How has AI investment changed in recent years?

Sources:
[Source 1] Page 248, Chunk 1
s investment section 
in the AI Index includes data on generative AI 
investments.
4.3 Investment
Corporate Investment
Figure 4.3.1 illustrates the trend in global corporate AI investment from 
2013 to 2024, including mergers and acquisitions, minority stakes, private 
investments, and public offerings. 
In 2024, the total investment grew to $252.3 billion, an increase of 25.5% 
from 2023. The most significant upturn occurred in private investment, 
which rose by 44.5% compared with the previous year, while mergers and 
acquisitions increased by 12.1%. Over the past decade, AI-related investments 
have increased nearly 

## Simple Evidence-Based Answer Without LLM

In [47]:
def answer_with_evidence_only(query, first_k=20, top_k=3):
    results = retrieve_with_rerank(query, first_k=first_k, top_k=top_k)

    print("Question:")
    print(query)
    print("\nMost relevant evidence:")
    print("=" * 120)

    for r in results:
        print(f"Source: Page {r['page']}, Chunk {r['chunk_id']}")
        print(f"Rerank Score: {r['rerank_score']:.4f}")
        print("-" * 120)
        print(r["text"][:1200])
        print("=" * 120)

In [49]:
answer_with_evidence_only(
    "How has AI investment changed in recent years?",
    first_k=20,
    top_k=3
)

Question:
How has AI investment changed in recent years?

Most relevant evidence:
Source: Page 248, Chunk 1
Rerank Score: 5.2594
------------------------------------------------------------------------------------------------------------------------
s investment section 
in the AI Index includes data on generative AI 
investments.
4.3 Investment
Corporate Investment
Figure 4.3.1 illustrates the trend in global corporate AI investment from 
2013 to 2024, including mergers and acquisitions, minority stakes, private 
investments, and public offerings. 
In 2024, the total investment grew to $252.3 billion, an increase of 25.5% 
from 2023. The most significant upturn occurred in private investment, 
which rose by 44.5% compared with the previous year, while mergers and 
acquisitions increased by 12.1%. Over the past decade, AI-related investments 
have increased nearly thirteenfold.
20.06
37.32
25.72
43.1
58.18
73.79
145.4
113.01
104.34
150.79
88.19
24.68
21.89
36.43
39.83
175.36
121.39
82.

## Add Retrieval Evaluation

In [52]:
eval_questions = [
    {
        "question": "How has AI investment changed in recent years?",
        "expected_keywords": ["investment", "private", "AI"]
    },
    {
        "question": "What does the report say about the cost of AI inference?",
        "expected_keywords": ["inference", "cost"]
    },
    {
        "question": "What are the trends in AI publications and patents?",
        "expected_keywords": ["publications", "patents"]
    },
    {
        "question": "How is AI being used in healthcare?",
        "expected_keywords": ["healthcare", "medical"]
    },
    {
        "question": "What risks or challenges are associated with AI adoption?",
        "expected_keywords": ["risk", "adoption"]
    }
]

eval_df = pd.DataFrame(eval_questions)
eval_df

,question,expected_keywords
0,How has AI investment changed in recent years?,"[investment, private, AI]"
1,What does the report say about the cost of AI ...,"[inference, cost]"
2,What are the trends in AI publications and pat...,"[publications, patents]"
3,How is AI being used in healthcare?,"[healthcare, medical]"
4,What risks or challenges are associated with A...,"[risk, adoption]"


In [54]:
def keyword_hit(text, keywords):
    text_lower = text.lower()
    return any(keyword.lower() in text_lower for keyword in keywords)


def evaluate_recall_at_k(eval_df, retrieve_func, k=5):
    hits = 0

    for _, row in eval_df.iterrows():
        query = row["question"]
        keywords = row["expected_keywords"]

        results = retrieve_func(query, top_k=k)
        combined_text = " ".join([r["text"] for r in results])

        if keyword_hit(combined_text, keywords):
            hits += 1

    return hits / len(eval_df)

In [65]:
recall_dense_5 = evaluate_recall_at_k(
    eval_df,
    retrieve_func=retrieve_dense_cosine,
    k=5
)

print("Dense Retrieval Recall@5:", recall_dense_5)

Dense Retrieval Recall@5: 1.0


In [58]:
def retrieve_rerank_wrapper(query, top_k=5):
    return retrieve_with_rerank(query, first_k=20, top_k=top_k)


recall_rerank_5 = evaluate_recall_at_k(
    eval_df,
    retrieve_func=retrieve_rerank_wrapper,
    k=5
)

print("Reranked Retrieval Recall@5:", recall_rerank_5)

Reranked Retrieval Recall@5: 1.0


## Add MRR Evaluation

In [61]:
def evaluate_mrr(eval_df, retrieve_func, k=5):
    reciprocal_ranks = []

    for _, row in eval_df.iterrows():
        query = row["question"]
        keywords = row["expected_keywords"]

        results = retrieve_func(query, top_k=k)

        rank_found = None

        for i, r in enumerate(results):
            if keyword_hit(r["text"], keywords):
                rank_found = i + 1
                break

        if rank_found is None:
            reciprocal_ranks.append(0)
        else:
            reciprocal_ranks.append(1 / rank_found)

    return sum(reciprocal_ranks) / len(reciprocal_ranks)

In [63]:
mrr_dense = evaluate_mrr(
    eval_df,
    retrieve_func=retrieve_dense_cosine,
    k=5
)

mrr_rerank = evaluate_mrr(
    eval_df,
    retrieve_func=retrieve_rerank_wrapper,
    k=5
)

print("Dense Retrieval MRR@5:", mrr_dense)
print("Reranked Retrieval MRR@5:", mrr_rerank)

Dense Retrieval MRR@5: 1.0
Reranked Retrieval MRR@5: 0.9


## Error Analysis

In [68]:
def keyword_hit_with_matches(text, keywords):
    text_lower = text.lower()
    matched = []

    for keyword in keywords:
        if keyword.lower() in text_lower:
            matched.append(keyword)

    return len(matched) > 0, matched


def evaluate_per_query(eval_df, retrieve_func, k=5):
    rows = []

    for _, row in eval_df.iterrows():
        query = row["question"]
        keywords = row["expected_keywords"]

        results = retrieve_func(query, top_k=k)

        first_hit_rank = None
        matched_keywords_at_hit = []
        hit_page = None
        hit_preview = None

        for i, r in enumerate(results):
            hit, matched = keyword_hit_with_matches(r["text"], keywords)

            if hit:
                first_hit_rank = i + 1
                matched_keywords_at_hit = matched
                hit_page = r["page"]
                hit_preview = r["text"][:250].replace("\n", " ")
                break

        reciprocal_rank = 0 if first_hit_rank is None else 1 / first_hit_rank

        rows.append({
            "question": query,
            "first_hit_rank": first_hit_rank,
            "reciprocal_rank": reciprocal_rank,
            "hit_page": hit_page,
            "matched_keywords": matched_keywords_at_hit,
            "hit_preview": hit_preview
        })

    return pd.DataFrame(rows)

In [70]:
dense_detail = evaluate_per_query(
    eval_df,
    retrieve_func=retrieve_dense_cosine,
    k=5
)

rerank_detail = evaluate_per_query(
    eval_df,
    retrieve_func=retrieve_rerank_wrapper,
    k=5
)

dense_detail

,question,first_hit_rank,reciprocal_rank,hit_page,matched_keywords,hit_preview
0,How has AI investment changed in recent years?,1,1.0,359,"[investment, AI]",358 Artificial Intelligence Index Report 2025 ...
1,What does the report say about the cost of AI ...,1,1.0,65,"[inference, cost]",Table of Contents 64 Artificial Intelligence I...
2,What are the trends in AI publications and pat...,1,1.0,43,"[publications, patents]",Table of Contents 42 Artificial Intelligence I...
3,How is AI being used in healthcare?,1,1.0,284,[medical],283 Artificial Intelligence Index Report 2025 ...
4,What risks or challenges are associated with A...,1,1.0,183,"[risk, adoption]",182 Artificial Intelligence Index Report 2025 ...


In [72]:
rerank_detail

,question,first_hit_rank,reciprocal_rank,hit_page,matched_keywords,hit_preview
0,How has AI investment changed in recent years?,1,1.0,248,"[investment, private, AI]",s investment section in the AI Index includes...
1,What does the report say about the cost of AI ...,1,1.0,65,"[inference, cost]",Table of Contents 64 Artificial Intelligence I...
2,What are the trends in AI publications and pat...,1,1.0,43,"[publications, patents]",Table of Contents 42 Artificial Intelligence I...
3,How is AI being used in healthcare?,2,0.5,317,[healthcare],316 Artificial Intelligence Index Report 2025 ...
4,What risks or challenges are associated with A...,1,1.0,183,"[risk, adoption]",182 Artificial Intelligence Index Report 2025 ...


In [74]:
comparison = dense_detail[["question", "first_hit_rank", "reciprocal_rank", "hit_page"]].merge(
    rerank_detail[["question", "first_hit_rank", "reciprocal_rank", "hit_page"]],
    on="question",
    suffixes=("_dense", "_rerank")
)

comparison["mrr_change"] = comparison["reciprocal_rank_rerank"] - comparison["reciprocal_rank_dense"]

comparison

,question,first_hit_rank_dense,reciprocal_rank_dense,hit_page_dense,first_hit_rank_rerank,reciprocal_rank_rerank,hit_page_rerank,mrr_change
0,How has AI investment changed in recent years?,1,1.0,359,1,1.0,248,0.0
1,What does the report say about the cost of AI ...,1,1.0,65,1,1.0,65,0.0
2,What are the trends in AI publications and pat...,1,1.0,43,1,1.0,43,0.0
3,How is AI being used in healthcare?,1,1.0,284,2,0.5,317,-0.5
4,What risks or challenges are associated with A...,1,1.0,183,1,1.0,183,0.0


In [76]:
comparison[comparison["mrr_change"] < 0]

,question,first_hit_rank_dense,reciprocal_rank_dense,hit_page_dense,first_hit_rank_rerank,reciprocal_rank_rerank,hit_page_rerank,mrr_change
3,How is AI being used in healthcare?,1,1.0,284,2,0.5,317,-0.5
